# Open Hit Expander + Boltz-2 — compound assessment (v0.7)

Enumerate analogues of a hit, triage them on CPU, cofold the survivors with Boltz-2 (predicted affinity and pose), then inspect poses, protein–ligand interactions and prediction stability, and export an assessment report.

**Predictions are hypotheses for prioritisation, not measured potencies.** Check the ranking against known actives for your target before relying on it.

| # | Section | Runs in | Needs Boltz-2 / GPU |
|---|---------|---------|---------------------|
| 1 | Configuration | both | no |
| 2 | Environment check | both | no |
| 3 | Hit expansion | new | no |
| 4 | Pre-filter | new | no |
| 5 | Prepare batch and dry run | new | no |
| 6 | Cofolding replicates (resumable) | new | **yes — the only inference step** |
| 7 | Load and aggregate results | both | no |
| 8 | Pose viewer | both | no |
| 9 | Interaction analysis (PLIP) | new, or reload with saved CSVs | no |
| 10 | Interaction plots | both | no |
| 11 | Prediction stability | both | no |
| 12 | Compound assessment report | both | no |
| 13 | Files written | — | — |

**Modes.** `RUN_MODE = "new"` runs sections 3–6 and then everything downstream. `RUN_MODE = "reload"` skips them and analyses a finished run named by `SAVED_TIMESTAMP`; it never launches Boltz-2. If a new run is interrupted, rerun section 6 and only the unfinished jobs are computed.

## 1. Configuration

All settings live in this one cell. Later cells only read them. Values marked *(new run)* are ignored in reload mode.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
# Run from the repository root:  %pip install -e .
import json
import os
import platform
import shutil
import subprocess
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

from open_hit_expander import (
    HitExpanderConfig, SCREENING_PRESETS, collect_boltz2_replicate_results,
    load_boltz2_run, plan_boltz2_replicates, prepare_boltz2_batch, run_boltz2_replicates,
)
from open_hit_expander.interactions import analyze_interactions, display_interaction_heatmap
from open_hit_expander.plotting import plot_interaction_summary
from open_hit_expander.prefilter import PrefilterConfig, apply_prefilter, export_prefilter
from open_hit_expander.report import build_compound_assessment
from open_hit_expander.stability import plot_stability
from open_hit_expander.visualization import complex_browser, save_complex_browser
from open_hit_expander.widgets import hit_expander_widget

# ── 1. Workflow ──────────────────────────────────────────────────────────────
RUN_MODE = "new"          # "new": enumerate and run Boltz-2  |  "reload": analyse a finished run
SAVED_TIMESTAMP = ""      # reload only, e.g. "20261013_111348_145796"
OUTPUT_DIRECTORY = Path("hit_expander_output")
N_REPLICATES = 2          # seeds per ligand: 1 = quick look, 5 = confirmation

# ── 2. Ligand, enumeration and pre-filter (new run) ──────────────────────────
INPUT_SMILES = "CN(C)C[C@@H](O)COc1ccc(Nc2nccc(Nc3cc(Cl)ccc3Cl)n2)cc1"
ENUMERATION = HitExpanderConfig()
PREFILTER = PrefilterConfig(                  # thresholds are illustrative: set them for your series
    # core_atoms={"mol_1": [0, 1, 2]},        # parent atom indices every analogue must keep
    # forbid_smarts=("[N+](=O)[O-]",),
    # max_candidates_per_parent=25,           # diversity-aware cap (selection="maxmin")
)

# ── 3. Target (new run) ──────────────────────────────────────────────────────
# Example target: canonical chain-A sequence of human CDK2, PDB 1H01. The deposited
# construct has a modified lysine (KCX); Boltz receives canonical Lys at that position.
PROTEIN_SEQUENCE = (
    "MENFQKVEKIGEGTYGVVYKARNKLTGEVVALKKIRLDTETEGVPSTAIREISLLKELNHPNIVKLLDVIHTENKLYLVF"
    "EFLHQDLKKFMDASALTGIPLPLIKSYLFQLLQGLAFCHSHRVLHRDLKPQNLLINTEGAIKLADFGLARAFGVPVRTYT"
    "HEVVTLWYRAPEILLGCKYYSTAVDIWSLGCIFAEMVTRRALFPGDSEIDQLFRIFRTLGTPDEVVWPGVTSMPDYKPSF"
    "PKWARQDFSKVVPPLDEDGRSLLSQMLHYDPNKRISAKAALAHPFFQDVTKPVPHLRL"
)
MSA_PATH = None           # None: generate once via the MSA server and cache it. Or Path(".../target.a3m")

# ── 4. Boltz-2 (new run) ─────────────────────────────────────────────────────
SCREENING_PRESET = "balanced"        # "quick" | "balanced" | "confirm"  (see SCREENING_PRESETS)
MAX_ANALOGUES = 25                   # cap on cofolded analogues; None only for small sets
USE_AFFINITY_MW_CORRECTION = False   # False keeps the uncorrected affinity head
ACCELERATOR, NUM_DEVICES = "gpu", 1  # Apple Silicon (MPS) settings below
NO_KERNELS, NUM_WORKERS, PREPROCESSING_THREADS = True, 0, 8

# ── 5. Analysis ──────────────────────────────────────────────────────────────
PROTEIN_CHAIN, LIGAND_CHAIN = "A", "B"
VIEWER_INLINE = False     # False: write complex_browser.html and open it in a browser (most reliable)
THREE_DMOL_JS = None      # optional path to a downloaded 3Dmol-min.js, for offline use of the viewer
SHOW_FIGURES = True

## 2. Environment check

Validates the settings, locates the Boltz-2 executable (new runs) and prints the versions in use.

In [ ]:
assert RUN_MODE in {"new", "reload"}, 'RUN_MODE must be "new" or "reload"'
assert N_REPLICATES >= 1, "N_REPLICATES must be at least 1"
if RUN_MODE == "reload" and not SAVED_TIMESTAMP:
    raise ValueError("Reload mode needs SAVED_TIMESTAMP, e.g. '20261013_111348_145796'")

BOLTZ_EXECUTABLE = None
if RUN_MODE == "new":
    # Resolve from the BOLTZ_EXECUTABLE override, PATH, or a separate conda env named "boltz2".
    BOLTZ_EXECUTABLE = os.environ.get("BOLTZ_EXECUTABLE") or shutil.which("boltz")
    if not BOLTZ_EXECUTABLE and shutil.which("conda"):
        found = subprocess.run(["conda", "run", "-n", "boltz2", "which", "boltz"],
                               capture_output=True, text=True)
        if found.returncode == 0:
            BOLTZ_EXECUTABLE = found.stdout.strip()
    if not BOLTZ_EXECUTABLE:
        raise FileNotFoundError("Boltz executable not found: activate its environment or set BOLTZ_EXECUTABLE")

def _version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not installed"

print(f"Mode: {RUN_MODE}" + (f"   Saved run: {SAVED_TIMESTAMP}" if RUN_MODE == "reload" else ""))
print(f"Python {platform.python_version()}   open-hit-expander {_version('open-hit-expander-boltz2')}   "
      f"RDKit {_version('rdkit')}   pandas {pd.__version__}")
print("Boltz executable:", BOLTZ_EXECUTABLE or "not needed in reload mode")

## 3. Hit expansion — new run only

Enumerates single-site analogues of the input structure. Click **Enumerate analogues** in the widget before continuing to section 4.

In [ ]:
RUN_STATE = {"run": None, "output_files": None}

if RUN_MODE == "new":
    display(hit_expander_widget(INPUT_SMILES, ENUMERATION, OUTPUT_DIRECTORY, state=RUN_STATE))
else:
    print("Reload mode: enumeration skipped.")

## 4. Pre-filter before cofolding — new run only

CPU-only triage relative to the parent: property drift, alerts the analogue *introduces*, core retention, Boltz-2 size limits and an optional diversity-aware cap. Every decision is written to `prefilter_log.csv`; the funnel below shows how many analogues survive each stage.

In [ ]:
if RUN_MODE == "new":
    if RUN_STATE.get("run") is None:
        raise RuntimeError("Click 'Enumerate analogues' in section 3 first")
    RUN_STATE["run"] = apply_prefilter(RUN_STATE["run"], PREFILTER)
    display(RUN_STATE["run"]["prefilter_funnel"])
    export_prefilter(RUN_STATE["run"], OUTPUT_DIRECTORY)
else:
    print("Reload mode: pre-filter skipped.")

## 5. Prepare the Boltz-2 batch and dry run — new run only

Writes one YAML job per ligand and builds the `boltz predict` command. The table below is a **dry run**: it counts jobs already finished on disk or restorable from the cross-run cache, and launches nothing.

In [ ]:
BOLTZ2_STATE = None

if RUN_MODE == "new":
    if RUN_STATE.get("run") is None:
        raise RuntimeError("Enumerate analogues (section 3) before preparing Boltz-2 jobs")
    BOLTZ2_STATE = prepare_boltz2_batch(
        output_directory=OUTPUT_DIRECTORY,
        hit_run=RUN_STATE["run"],
        protein_sequence=PROTEIN_SEQUENCE,
        boltz_executable=BOLTZ_EXECUTABLE,
        msa_path=MSA_PATH,
        max_analogues=MAX_ANALOGUES,
        screening_preset=SCREENING_PRESET,
        screening_presets=SCREENING_PRESETS,
        diffusion_samples=SCREENING_PRESETS[SCREENING_PRESET]["diffusion_samples"],
        accelerator=ACCELERATOR,
        devices=NUM_DEVICES,
        no_kernels=NO_KERNELS,
        num_workers=NUM_WORKERS,
        preprocessing_threads=PREPROCESSING_THREADS,
        affinity_mw_correction=USE_AFFINITY_MW_CORRECTION,
    )
    manifest = BOLTZ2_STATE["manifest"]
    shown = ["job_id", "source_id", "is_parent", "transformation", "ligand_smiles",
             "morgan_tanimoto", "boltz_atom_count", "prefilter_flags"]
    display(manifest[[c for c in shown if c in manifest.columns]])
    print(f"Jobs: {len(manifest)}   Preset: {BOLTZ2_STATE['screening_preset']}   "
          f"Settings: {BOLTZ2_STATE['screening_settings']}")
    print("Run directory:", Path(BOLTZ2_STATE["root"]).resolve())
    print("Command:", " ".join(BOLTZ2_STATE["command"]))

    # Record the settings that produced this run, next to the run itself.
    (Path(BOLTZ2_STATE["root"]) / "notebook_settings.json").write_text(json.dumps({
        "input_smiles": INPUT_SMILES, "n_replicates": N_REPLICATES,
        "screening_preset": SCREENING_PRESET, "max_analogues": MAX_ANALOGUES,
        "affinity_mw_correction": USE_AFFINITY_MW_CORRECTION,
        "accelerator": ACCELERATOR, "msa_path": str(MSA_PATH) if MSA_PATH else None,
        "prefilter": repr(PREFILTER),
        "versions": {"open_hit_expander": _version("open-hit-expander-boltz2"),
                     "rdkit": _version("rdkit"), "python": platform.python_version()},
    }, indent=2))

    display(plan_boltz2_replicates(BOLTZ2_STATE, n_replicates=N_REPLICATES))
else:
    print("Reload mode: no new job directory created.")

## 6. Cofolding and potency replicates — new run only

**The only inference step.** Each seed produces a complex and one aggregate affinity prediction per ligand. The run is resumable per job: after an interruption, rerun this cell and only unfinished jobs are computed. Finished predictions are also cached across runs, and the target MSA is generated once and reused. Reload mode never launches Boltz-2.

In [ ]:
if RUN_MODE == "new":
    replicate_predictions, replicate_status = run_boltz2_replicates(BOLTZ2_STATE, n_replicates=N_REPLICATES)
    display(replicate_status)
else:
    print("Reload mode: no inference performed.")

## 7. Load and aggregate results

Reads saved predictions only. `predicted_pic50` is the median over replicates. Each ligand also carries its **representative** pose (the replicate closest to the median) and its **best** pose (the highest-pIC50 replicate); with two replicates these may coincide.

In [ ]:
if RUN_MODE == "reload":
    BOLTZ2_STATE, results, ranked = load_boltz2_run(
        SAVED_TIMESTAMP, output_directory=OUTPUT_DIRECTORY,
        reparse=True,  # repairs legacy missing_prediction_directory errors
    )
else:
    results, ranked = collect_boltz2_replicate_results(BOLTZ2_STATE)

RUN_ROOT = Path(BOLTZ2_STATE["root"])
summary_columns = [
    "job_id", "is_parent", "predicted_pic50", "mean_pic50", "sd_pic50", "min_pic50", "max_pic50",
    "median_delta_pic50", "sd_delta_pic50", "fraction_predicted_improvement", "n_success",
    "representative_replicate", "representative_seed", "representative_pic50", "cofold_status",
]
display(results[[c for c in summary_columns if c in results.columns]])
print("Run:", RUN_ROOT)
print("Successful poses:", int(results["structure_path"].notna().sum()), "of", len(results))

## 8. Pose viewer

Overlays saved poses after aligning protein Cα atoms; navigation performs no inference. The viewer is written to `complex_browser.html` in the run folder and opened in your browser, because the inline notebook version can render as a blank box when the payload is large or the 3Dmol.js script is blocked. Set `VIEWER_INLINE = True` to try the inline version.

In [ ]:
pose_columns = ["job_id", "predicted_pic50", "representative_pic50", "best_pic50",
                "best_replicate", "best_structure_path"]
display(results[[c for c in pose_columns if c in results.columns]])

if VIEWER_INLINE:
    display(complex_browser(results, ligand_chain=LIGAND_CHAIN, protein_chain=PROTEIN_CHAIN))
else:
    viewer_file = save_complex_browser(
        results, RUN_ROOT / "complex_browser.html",
        ligand_chain=LIGAND_CHAIN, protein_chain=PROTEIN_CHAIN,
        three_dmol_js=THREE_DMOL_JS, open_in_browser=True)
    print("Viewer:", viewer_file.resolve(), f"({viewer_file.stat().st_size / 1e6:.1f} MB)")

## 9. Interaction analysis (PLIP)

Analyses only the representative pose of each ligand and saves results inside the selected run, so runs never overwrite each other. In reload mode previously saved PLIP CSVs are reused. This step does not run Boltz-2.

In [ ]:
ANALYSIS_DIR = RUN_ROOT / "interaction_analysis"
RUN_PLIP = True # (RUN_MODE == "new")   # set True to (re)analyse in reload mode

if RUN_PLIP:
    if not shutil.which("plip"):
        raise RuntimeError("PLIP is not on this kernel's PATH")
    interaction_details, interaction_fingerprints, interaction_status, heatmap_path = analyze_interactions(
        results, output_dir=ANALYSIS_DIR,
        protein_chain=PROTEIN_CHAIN, ligand_chain=LIGAND_CHAIN, plip_executable="plip",
    )
    display(interaction_status)
    display_interaction_heatmap(interaction_details, interaction_status)
else:
    print("PLIP skipped: saved interaction CSVs are used if present. Set RUN_PLIP = True to reanalyse.")

## 10. Interaction plots

Fingerprint, interaction counts, residue conservation and fingerprint similarity. Figures are also saved as PNG in `interaction_analysis/figures/`.

In [ ]:
if (ANALYSIS_DIR / "interaction_details.csv").exists():
    interaction_figures = plot_interaction_summary(
        details=ANALYSIS_DIR, results=results,
        ligand_sort="predicted_pic50", residue_sort="sequence",
        show=SHOW_FIGURES, save_dir=ANALYSIS_DIR / "figures",
    )
else:
    print("No interaction analysis found; run section 9 first.")

## 11. Prediction stability

Uses saved replicates only. Each replicate is a dot, the tick is the median, and the line spans minimum to maximum. The second plot shows the paired change against the parent and whether it holds in every replicate. With a single replicate there is nothing to compare. Figures are also saved in `figures/` in the run folder.

In [ ]:
if (RUN_ROOT / "replicate_predictions.csv").exists():
    stability_columns = ["job_id", "median_pic50", "mean_pic50", "sd_pic50", "median_delta_pic50",
                         "sd_delta_pic50", "fraction_predicted_improvement", "n_success", "n_paired"]
    display(results[[c for c in stability_columns if c in results.columns]])
    n_seeds = pd.read_csv(RUN_ROOT / "replicate_predictions.csv")["seed"].nunique()
    if n_seeds > 1:
        stability_figures = plot_stability(RUN_ROOT, show=SHOW_FIGURES, save_dir=RUN_ROOT / "figures")
    else:
        print("Only one replicate: between-seed spread cannot be estimated.")
else:
    print("No replicate_predictions.csv found in the run folder.")

## 12. Compound assessment report

Builds the sortable compound ranking, potency-stability plots and per-ligand interaction changes from the existing `results` table and saved PLIP outputs. No Boltz-2 inference is performed.

In [ ]:
REPORT_DIR = RUN_ROOT / "compound_assessment"
compound_summary, compound_report = build_compound_assessment(
    results,
    details=ANALYSIS_DIR / "interaction_details.csv",
    status=ANALYSIS_DIR / "interaction_status.csv",
    replicate_predictions=RUN_ROOT / "replicate_predictions.csv",
    output_dir=REPORT_DIR,
)
display(compound_summary)
print("HTML report:", compound_report.resolve())
display(HTML(f'<a href="{compound_report.resolve().as_uri()}" target="_blank">Open compound assessment</a>'))

## 13. Files written

Everything for one run lives under `hit_expander_output/boltz2_cofold/<timestamp>/` (`RUN_ROOT`); new and reloaded runs use the same layout. There is no Boltz-2 call after section 6.

| Location | Contents |
|----------|----------|
| `hit_expander_output/prefilter_*.csv` | decision log, funnel and rejected analogues from section 4 |
| `RUN_ROOT/jobs/`, `notebook_settings.json` | Boltz-2 input YAMLs; the settings that produced the run |
| `RUN_ROOT/replicates/replicate_NN_seed_S/` | per-seed `results/`, `boltz.log`, `job_status.csv`, `replicate_results.csv` |
| `RUN_ROOT/replicate_predictions.csv`, `replicate_status.csv` | all replicate predictions and per-replicate status |
| `RUN_ROOT/complex_browser.html` | standalone pose viewer |
| `RUN_ROOT/interaction_analysis/` | PLIP details, status, fingerprints and `figures/` |
| `RUN_ROOT/figures/` | stability plots (PNG) |
| `RUN_ROOT/compound_assessment/` | final ranking table and HTML report (`compound_summary.csv` includes `prefilter_flags`) |
| `hit_expander_output/cofold_cache/`, `msa_cache/` | cross-run prediction cache and reusable target MSA |